In [6]:
import pandas as pd
import numpy as np
import random

In [7]:
np.random.seed(42)
random.seed(42)

In [8]:
#TABELA AGENTES

df_agentes = pd.DataFrame({"id_agente": [3331, 3332, 3333, 3334, 3335, 3336],
                           "nome": ["Ana", "Emily", "Lucas", "Vitor", "Gabriel", "Lara"],
                           "turno": ["Manhã", "Tarde", "Manhã", "Noite", "Manhã", "Tarde"]
                           })
df_agentes

,id_agente,nome,turno
0,3331,Ana,Manhã
1,3332,Emily,Tarde
2,3333,Lucas,Manhã
3,3334,Vitor,Noite
4,3335,Gabriel,Manhã
5,3336,Lara,Tarde


In [9]:
#TABELA ATENDIMENTOS

N = 1500

data_abertura = pd.Timestamp('2025-01-01')
datas = data_abertura + pd.to_timedelta(np.random.randint(0, 181, N), unit='D')
prazos = {'pagamento': 24, 'bug': 48, 'dúvida': 8, 'cancelamento': 24, 'acesso': 12, 'lentidão': 36 }
categoria = np.random.choice(["pagamento", "bug", "dúvida", "cancelamento", "acesso", "lentidão"], N)
status_ticket= np.random.choice(["aberto", "resolvido", "não resolvido"], N, p = [0.15, 0.65, 0.20])

df_atendimentos = pd.DataFrame({
    'id_chamado': range(1, N+1),
    'id_agente': np.random.choice(df_agentes['id_agente'], N),
    'categoria': categoria,
    'data_abertura': datas,
    'status_ticket': status_ticket,
    'prazo_atendimento': pd.Series(categoria).map(prazos),
    'data_fechamento': np.where( status_ticket == 'aberto', pd.NaT, datas + pd.to_timedelta( np.random.randint (0, 48, N), unit= 'h')),
})

df_atendimentos['data_fechamento'] = pd.to_datetime(df_atendimentos['data_fechamento'])


In [10]:
df_atendimentos.head(10)

,id_chamado,id_agente,categoria,data_abertura,status_ticket,prazo_atendimento,data_fechamento
0,1,3336,bug,2025-04-13,resolvido,48,2025-04-13 10:00:00
1,2,3333,lentidão,2025-06-29,resolvido,36,2025-06-30 10:00:00
2,3,3334,acesso,2025-04-03,aberto,12,NaT
3,4,3334,pagamento,2025-01-15,não resolvido,24,2025-01-16 17:00:00
4,5,3336,pagamento,2025-04-17,resolvido,24,2025-04-17 12:00:00
5,6,3331,acesso,2025-03-13,resolvido,12,2025-03-14 11:00:00
6,7,3331,lentidão,2025-01-21,resolvido,36,2025-01-21 12:00:00
7,8,3332,pagamento,2025-04-13,resolvido,24,2025-04-13 11:00:00
8,9,3331,pagamento,2025-05-02,aberto,24,NaT
9,10,3332,pagamento,2025-03-16,resolvido,24,2025-03-16 09:00:00


In [11]:
# TABELA FEEDBACKS

df_resolvidos = df_atendimentos[df_atendimentos['status_ticket'] == 'resolvido'].copy()
len(df_resolvidos)

n_feedbacks = int(0.6 * len(df_resolvidos))
ids_com_feedback = random.sample(list(df_resolvidos['id_chamado']), n_feedbacks)

comentarios_negativos = ['Péssimo', 'Não resolveu meu problema']
comentarios_neutros   = ['Regular', 'OK', 'Nada a reclamar']
comentarios_positivos = ['Ótimo', 'Problema resolvido', 'Agente atencioso', 'Excelente']

def sortear_comentario(nota):
    if nota <= 2:
        return random.choice(comentarios_negativos)
    elif nota == 3:
        return random.choice(comentarios_neutros)
    else:
        return random.choice(comentarios_positivos)

df_feedbacks = pd.DataFrame({
      "id_feedback": range(1, n_feedbacks + 1),
      "id_chamado": ids_com_feedback,
      "nota": np.random.randint(1, 6, n_feedbacks),
      "comentario": np.random.choice(["Muito bom", "Bom", "Regular", "Ruim", "Muito ruim"], n_feedbacks)
})

df_feedbacks['comentario'] = df_feedbacks['nota'].map(sortear_comentario)

df_feedbacks.head()

,id_feedback,id_chamado,nota,comentario
0,1,1032,4,Ótimo
1,2,186,1,Não resolveu meu problema
2,3,35,1,Não resolveu meu problema
3,4,1184,3,Regular
4,5,446,4,Excelente


In [12]:
#SALVAR CSVs

df_agentes.to_csv('agentes.csv', index=False)
df_atendimentos.to_csv('atendimentos.csv', index=False)
df_feedbacks.to_csv('feedbacks.csv', index=False)

from google.colab import files
files.download('agentes.csv')
files.download('atendimentos.csv')
files.download('feedbacks.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import sqlite3

conn = sqlite3.connect('suporte.db')

df_agentes.to_sql('agentes', conn, if_exists='replace', index=False)
df_atendimentos.to_sql('atendimentos', conn, if_exists='replace', index=False)
df_feedbacks.to_sql('feedbacks', conn, if_exists='replace', index=False)

print('Tabelas criadas com sucesso')

Tabelas criadas com sucesso


## Pergunta 1 — Qual é a taxa de resolução geral do suporte?

In [14]:
query = """
SELECT
    status_ticket,
    COUNT(*) AS quantidade,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM atendimentos), 1) AS percentual
FROM atendimentos
GROUP BY status_ticket
ORDER BY quantidade DESC
"""

pd.read_sql(query, conn)

,status_ticket,quantidade,percentual
0,resolvido,966,64.4
1,não resolvido,313,20.9
2,aberto,221,14.7


## Pergunta 2 — Qual categoria de problema tem maior volume?

In [15]:
query = """
SELECT
    categoria,
    COUNT(*) AS quantidade,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM atendimentos), 1) AS percentual
FROM atendimentos
GROUP BY categoria
ORDER BY quantidade DESC
"""

pd.read_sql(query, conn)

,categoria,quantidade,percentual
0,cancelamento,265,17.7
1,acesso,265,17.7
2,bug,259,17.3
3,lentidão,249,16.6
4,pagamento,244,16.3
5,dúvida,218,14.5


## Pergunta 3 — Qual o tempo médio de atendimento por categoria?

In [16]:
query = """
SELECT
    categoria,
    COUNT(*) AS quantidade,
    ROUND(AVG((JULIANDAY(data_fechamento) - JULIANDAY(data_abertura)) * 24), 1) AS tempo_medio_horas
FROM atendimentos
WHERE status_ticket != 'aberto'
GROUP BY categoria
ORDER BY quantidade DESC
"""

pd.read_sql(query, conn)

,categoria,quantidade,tempo_medio_horas
0,acesso,236,22.8
1,cancelamento,226,22.4
2,bug,220,23.6
3,lentidão,210,23.1
4,pagamento,199,23.7
5,dúvida,188,23.9


## Pergunta 4 — Desempenho por agente

In [17]:
query = """
SELECT
    agentes.nome,
    COUNT(*) AS total_atendimentos,
    ROUND(SUM(CASE WHEN status_ticket = 'resolvido' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS taxa_resolucao,
    ROUND(AVG(feedbacks.nota), 1) AS nota_media
FROM atendimentos
JOIN agentes ON atendimentos.id_agente = agentes.id_agente
LEFT JOIN feedbacks ON atendimentos.id_chamado = feedbacks.id_chamado
GROUP BY agentes.nome

"""
pd.read_sql(query, conn)

,nome,total_atendimentos,taxa_resolucao,nota_media
0,Ana,242,61.2,2.9
1,Emily,243,63.4,3.0
2,Gabriel,252,65.5,3.0
3,Lara,253,63.2,3.0
4,Lucas,250,66.0,2.9
5,Vitor,260,66.9,2.9


## Pergunta 5 — satisfação média por categoria.

In [18]:
query = """
SELECT
    atendimentos.categoria,
    ROUND(AVG(feedbacks.nota), 1) AS nota_media
FROM atendimentos
LEFT JOIN feedbacks ON atendimentos.id_chamado = feedbacks.id_chamado
GROUP BY atendimentos.categoria
ORDER BY nota_media DESC

"""
pd.read_sql(query, conn)

,categoria,nota_media
0,lentidão,3.1
1,pagamento,3.0
2,cancelamento,3.0
3,acesso,3.0
4,dúvida,2.8
5,bug,2.7


## Pergunta 6 — tendência mensal de volume.

In [20]:
query = """
SELECT
    strftime('%Y-%m', data_abertura) AS mes,
    COUNT(*) AS quantidade
FROM atendimentos
GROUP BY mes
ORDER BY mes

"""
pd.read_sql(query, conn)

,mes,quantidade
0,2025-01,251
1,2025-02,247
2,2025-03,229
3,2025-04,263
4,2025-05,267
5,2025-06,243


In [21]:
import os
os.makedirs('perguntas_de_negocio', exist_ok=True)

In [22]:
with open('perguntas_de_negocio/01_taxa_resolucao.sql', 'w') as f:
    f.write("""SELECT
    status_ticket,
    COUNT(*) AS quantidade,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM atendimentos), 1) AS percentual
FROM atendimentos
GROUP BY status_ticket
ORDER BY quantidade DESC""")

with open('perguntas_de_negocio/02_volume_por_categoria.sql', 'w') as f:
    f.write("""SELECT
    categoria,
    COUNT(*) AS quantidade,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM atendimentos), 1) AS percentual
FROM atendimentos
GROUP BY categoria
ORDER BY quantidade DESC""")

with open('perguntas_de_negocio/03_tempo_medio_atendimento.sql', 'w') as f:
    f.write("""SELECT
    categoria,
    COUNT(*) AS quantidade,
    ROUND(AVG((JULIANDAY(data_fechamento) - JULIANDAY(data_abertura)) * 24), 1) AS tempo_medio_horas
FROM atendimentos
WHERE status_ticket != 'aberto'
GROUP BY categoria
ORDER BY quantidade DESC""")

with open('perguntas_de_negocio/04_desempenho_por_agente.sql', 'w') as f:
    f.write("""SELECT
    agentes.nome,
    COUNT(*) AS total_atendimentos,
    ROUND(SUM(CASE WHEN status_ticket = 'resolvido' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS taxa_resolucao,
    ROUND(AVG(feedbacks.nota), 1) AS nota_media
FROM atendimentos
JOIN agentes ON atendimentos.id_agente = agentes.id_agente
LEFT JOIN feedbacks ON atendimentos.id_chamado = feedbacks.id_chamado
GROUP BY agentes.nome""")

with open('perguntas_de_negocio/05_satisfacao_por_categoria.sql', 'w') as f:
    f.write("""SELECT
    atendimentos.categoria,
    ROUND(AVG(feedbacks.nota), 1) AS nota_media
FROM atendimentos
LEFT JOIN feedbacks ON atendimentos.id_chamado = feedbacks.id_chamado
GROUP BY atendimentos.categoria
ORDER BY nota_media DESC""")

with open('perguntas_de_negocio/06_tendencia_mensal.sql', 'w') as f:
    f.write("""SELECT
    STRFTIME('%Y-%m', data_abertura) AS mes,
    COUNT(*) AS quantidade
FROM atendimentos
GROUP BY mes
ORDER BY mes""")

print('✅ Queries salvas com sucesso')



✅ Queries salvas com sucesso


In [26]:
from google.colab import files

# CSVs
files.download('agentes.csv')
files.download('atendimentos.csv')
files.download('feedbacks.csv')

# Queries
files.download('perguntas_de_negocio/01_taxa_resolucao.sql')
files.download('perguntas_de_negocio/02_volume_por_categoria.sql')
files.download('perguntas_de_negocio/03_tempo_medio_atendimento.sql')
files.download('perguntas_de_negocio/04_desempenho_por_agente.sql')
files.download('perguntas_de_negocio/05_satisfacao_por_categoria.sql')
files.download('perguntas_de_negocio/06_tendencia_mensal.sql')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>